---
title: Model Alerts
subtitle: Which changes deserve attention?
execute:
  warning: true
---
<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->

A new image contains a bright spot. Is it a changing astronomical source,
a detector defect, or the imperfect cancellation of a steady star? Model
Alerts follows that decision from electrons in two exposures to the list a
person reviews. We generate independent fields, detect positive changes
with **xPois**, and train **xScan** from random initialization. **xFit** supplies
an additional measured-feature investigation.

[View the source notebook in NVIDIA/cuPhoton](https://github.com/NVIDIA/cuPhoton/blob/main/examples/model-alerts/model-alerts.ipynb).

Every image, source, defect and noise realization is generated here from
declared analytic models. No observed survey images or pretrained models
enter the experiment. The results describe this modeled population. They do
not establish survey performance or calibrated probabilities for the sky.

Read the expanded **Science Explanations** for assumptions, units and
limitations. Follow a terminology link for a definition; keyboard focus and
pointer hover also reveal it in the HTML. All plot controls explore saved
arrays and scores in your browser. They do not train a new classifier.

## The decision comes before the classifier

We call a detected PSF-like appearance or brightening **positive**. Detector
artifacts, noise peaks and <a class="model-alerts-glossary-term" href="#model-alerts-glossary-residual">residual</a> light from static sources are **negative**.
The first task is to define those words, before showing a model any labels.

An untrailed moving source is a genuine astrophysical challenge. We report
it separately, rather than teaching the model that all motion is bogus.
Overlapping changes and artifacts receive ambiguity flags. The complete
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">candidate</a> inventory retains them, while primary binary metrics state their
coverage after excluding <a class="model-alerts-glossary-term" href="#model-alerts-glossary-ambiguous">ambiguous</a> associations and these challenges.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

A label answers a scientific question about a detected <a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">candidate</a>, not about
an arbitrary image location. The distribution is conditional on passing the
positive-detection rule. Pair classification therefore operates on
subtraction-selected candidates even though its input omits the difference
channel. Fading sources require a separate negative-polarity detection and
labeling extension; positive-only recovery does not measure that task.

</details>

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
from bokeh.io import output_notebook, show
from bokeh.resources import INLINE

from generator import Design, psf
from reference import execute_reference
from views import candidate_explorer, performance_diagnostics, training_diagnostics
from reporting import table, model_rows

output_notebook(resources=INLINE)
root = Path(os.environ.get("CUPHOTON_MODEL_ALERTS_OUTPUT", "results"))
profile = os.environ.get("CUPHOTON_MODEL_ALERTS_PROFILE", "quick-smoke")
print("Profile:", profile)
print("Quick smoke exercises APIs; scientific performance requires full reference.")

## Count electrons, then add independent noise

Our detector has unit gain: image values are electrons per pixel. A star
has total expected flux $F$ electrons and a pixel-integrated circular Gaussian
PSF. Reference seeing has $\sigma=0.9$–$1.2$ pixels; ordinary search seeing
has $\sigma=1.2$–$1.9$ pixels. Background is uniform within an exposure and
varies from 60 to 140 electrons per pixel. <a class="model-alerts-glossary-term" href="#model-alerts-glossary-read-noise">Read noise</a> is 4 electrons RMS.
These are teaching choices, not instrument measurements.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

For pixel $p$, the expectation is

<div class="model-alerts-equation" role="region" aria-label="Scientific equation" tabindex="0">

$$\mu_p=b+\sum_j F_j\int_{p}P(x-x_j,y-y_j;\sigma_j)\,dx\,dy.$$

</div>

The PSF has units pixel$^{-2}$ and integrates to one on the infinite plane.
The generator integrates it over each pixel using error functions rather
than sampling only the center. Exposures obey

<div class="model-alerts-equation" role="region" aria-label="Scientific equation" tabindex="0">

$$I_p\sim\operatorname{Poisson}(\mu_p)+\mathcal N(0,\sigma_r^2),\qquad
\operatorname{Var}(I_p)=\mu_p+\sigma_r^2.$$

</div>

The Poisson term is electron counting noise. <a class="model-alerts-glossary-term" href="#model-alerts-glossary-read-noise">Read noise</a> is independent of it
and of every other exposure. Finite image boundaries can truncate <a class="model-alerts-glossary-term" href="#model-alerts-glossary-flux">flux</a>;
injected source positions leave a declared margin. There is no saturation,
clipping or masked-pixel generator signature. Registered detector-plane
coordinates suffice here; no observed WCS or calibration product is used.

</details>

In [ ]:
design = Design()
y, x = np.indices((71, 71))
unit_psf = psf((71, 71), 34.2, 35.3, 1.5)
print("Integrated finite-field PSF flux:", unit_psf.sum())
print("Pixel centroid:", (unit_psf*x).sum(), (unit_psf*y).sum())
print("Declared generator:", design)

## Independent fields own their repeated observations

The <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">reference</a> design starts with 300 fields and three search epochs per
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-field">field</a>: 180 training fields, 60 validation fields and 60 reserved test fields.
Each <a class="model-alerts-glossary-term" href="#model-alerts-glossary-field">field</a> owns its stars, template noise, persistent defects and repeated
epochs. The specified shift adds 60 independent fields. A development pilot
runs before any reserved test <a class="model-alerts-glossary-term" href="#model-alerts-glossary-field">field</a> is generated.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

Repeated exposures of one <a class="model-alerts-glossary-term" href="#model-alerts-glossary-field">field</a> share sky structure and a noisy template.
Splitting their candidates independently would let the learner recognize the
same structure on both sides of an evaluation boundary. We <a class="model-alerts-glossary-term" href="#model-alerts-glossary-split">split</a> fields
first, then generate their observations from separate deterministic streams.
Native <a class="model-alerts-glossary-term" href="#model-alerts-glossary-provenance">provenance</a> guards reject field identities crossing splits. We also
audit seeds and duplicate <a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">candidate</a> bytes. Different row IDs do not establish
independence. If the primary validation field-bootstrap interval has
half-width above five percentage points, we double the development design
and reassess before freezing final counts and seed blocks.

</details>

## Create the residual that selects candidates

xPois fits a Gaussian-polynomial convolution <a class="model-alerts-glossary-term" href="#model-alerts-glossary-kernel">kernel</a> and differential
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-background">background</a> from measured images. It does not use injected source locations
as fitting masks. The residual is search minus the matched <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">reference</a>.
We filter it with a fixed Gaussian of $\sigma=1.4$ pixels, then select local
positive maxima above the development detection <a class="model-alerts-glossary-term" href="#model-alerts-glossary-threshold">threshold</a>. Cutouts are
centered on those measured maxima.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

The image model and <a class="model-alerts-glossary-term" href="#model-alerts-glossary-residual">residual</a> convention are

<div class="model-alerts-equation" role="region" aria-label="Scientific equation" tabindex="0">

$$S\simeq R\otimes K+B,\qquad D=S-(R\otimes K+B).$$

</div>

Independent gain/read-noise/<a class="model-alerts-glossary-term" href="#model-alerts-glossary-background">background</a> calibration and measured source
intensity estimate input variances. For a fixed fitted <a class="model-alerts-glossary-term" href="#model-alerts-glossary-kernel">kernel</a>,

<div class="model-alerts-equation" role="region" aria-label="Scientific equation" tabindex="0">

$$V_D\simeq V_S+V_R\otimes K^2,\quad
q=\frac{D\otimes H}{\sqrt{V_D\otimes H^2}}.$$

</div>

$q$ is a marginal detection diagnostic. It omits spatial covariance and
kernel-fit uncertainty; it is not a calibrated Gaussian significance. The
threshold is four in this diagnostic and the <a class="model-alerts-glossary-term" href="#model-alerts-glossary-association">association</a> radius is three
pixels, both declared development choices and frozen before final testing.
A complete stamp/<a class="model-alerts-glossary-term" href="#model-alerts-glossary-kernel">kernel</a> footprint is required. The same selection rule is
used for pair, triplet, <a class="model-alerts-glossary-term" href="#model-alerts-glossary-baseline">baseline</a> and fusion experiments.

</details>

## Truth labels detections; it never selects the easy ones

After detection, we associate peaks to injected changes within the frozen
radius. Unmatched detections remain negative examples. Bad pixels, cosmic
rays, registration errors and local <a class="model-alerts-glossary-term" href="#model-alerts-glossary-psf">PSF</a> mismatch all contribute candidates.
A real change overlapping a detector artifact, or several associated real
changes, is flagged as <a class="model-alerts-glossary-term" href="#model-alerts-glossary-ambiguous">ambiguous</a>.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

Position <a class="model-alerts-glossary-term" href="#model-alerts-glossary-association">association</a> is an evaluation convention with finite coverage. A
chance noise maximum near an injection can be associated; a badly displaced
real peak can fall outside the radius. Report injection recovery, <a class="model-alerts-glossary-term" href="#model-alerts-glossary-ambiguous">ambiguous</a>
counts and excluded challenge counts alongside classification metrics. A
positive detected near a static star may include overlapping static <a class="model-alerts-glossary-term" href="#model-alerts-glossary-residual">residual</a>
light; the overlap flag makes that limitation visible. Do not use truth
coordinates, family, filenames, <a class="model-alerts-glossary-term" href="#model-alerts-glossary-field">field</a> identity or split assignment as model
features.

</details>

## Build a native xScan dataset

The canonical arrays are `<a class="model-alerts-glossary-term" href="#model-alerts-glossary-search">search</a>`, `template`, `difference`, binary labels
and split assignments, with <a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">candidate</a> metadata stored separately. Raw
cutouts remain in <a class="model-alerts-glossary-term" href="#model-alerts-glossary-electron">electron</a> units for inspection. Model inputs use shared
search/<a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">reference</a> centering and scaling fitted only on eligible training
rows; difference scaling is also fitted only on training rows.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

A <a class="model-alerts-glossary-term" href="#model-alerts-glossary-normalization">normalization</a> estimate is itself a learned quantity. Fitting it on all
rows would expose held-out data even without their labels. The fitted
training constants are saved and applied unchanged to validation, test and
shifted fields. Pair and triplet use the same detection pool and compatible
channel scales. xFit receives corresponding <a class="model-alerts-glossary-term" href="#model-alerts-glossary-variance">variance</a> divided by the square
of the difference-channel scale. Exact-stamp hashes are calculated in the
units actually passed to the native <a class="model-alerts-glossary-term" href="#model-alerts-glossary-fusion">fusion</a> workflow.

</details>

In [ ]:
# The reference helper executes generation, native guards, pilot training,
# required controls, precision assessment, freeze and final inference.
# The outer runner holds the shared GPU lease for this entire call.
reference = execute_reference(root, profile=profile,
    device=os.environ.get("CUPHOTON_MODEL_ALERTS_DEVICE", "cuda:0"))
table(reference["design"], caption="Executed design")
table([dict(population=k, **v) for k,v in reference["coverage"].items()],
      caption="Detection and metric coverage")

## Learn from images, from random weights

We compare native xScan pair and triplet transformer models initialized
from scratch. The pair sees search and <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">reference</a>; the triplet also sees the
xPois <a class="model-alerts-glossary-term" href="#model-alerts-glossary-residual">residual</a>. Three initialization seeds are shown separately for each
principal model. The same fields, candidates and optimization settings
support a paired comparison.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

Training minimizes binary cross entropy from logits $z_i$,

<div class="model-alerts-equation" role="region" aria-label="Scientific equation" tabindex="0">

$$\mathcal L=-\frac1n\sum_i[y_i\log\sigma(z_i)+(1-y_i)\log(1-\sigma(z_i))].$$

</div>

The sigmoid converts a <a class="model-alerts-glossary-term" href="#model-alerts-glossary-logit">logit</a> to a score between zero and one. It does not
make that <a class="model-alerts-glossary-term" href="#model-alerts-glossary-score">score</a> a calibrated astrophysical probability. Checkpoints are
selected using validation loss. Our teaching configuration uses three
encoder stages with embedding dimensions 16, 32 and 64, one block per stage,
12 epochs and Adam-style native optimizer settings. These are example
choices, not a reproduction of a paper's published accuracy. Quick smoke
uses one <a class="model-alerts-glossary-term" href="#model-alerts-glossary-epoch">epoch</a> and fewer fields and cannot substantiate performance.

</details>

In [ ]:
table([dict(model=f"{r['mode']}-{r['seed']}", threshold=r["threshold"],
            validation_auc=r["validation_roc_auc"], **r["validation"])
       for r in reference["training"]], caption="Validation operating points")
show(training_diagnostics(root, reference))

## Can a simple measured statistic explain the result?

SNR-only logistic regression is the first <a class="model-alerts-glossary-term" href="#model-alerts-glossary-baseline">baseline</a>. A second baseline adds
positive core concentration, negative <a class="model-alerts-glossary-term" href="#model-alerts-glossary-residual">residual</a> fraction, peak-to-RMS ratio
and search/<a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">reference</a> correlation. Mask fraction and clipping fraction are
explicitly constant zero. Feature standardization and logistic coefficients
are fitted on training rows only.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

A high <a class="model-alerts-glossary-term" href="#model-alerts-glossary-score">score</a> can reflect a trivial generator cue rather than source
morphology. The baselines expose how much signal comes from brightness or
simple shape statistics. No family or injection-SNR column enters them.
The expected injection <a class="model-alerts-glossary-term" href="#model-alerts-glossary-snr">SNR</a> is log-uniform from four to 35 under an
approximate background-dominated Gaussian calculation; measured detection
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-snr">SNR</a> differs because of source noise, fitted subtraction and selection.
Performance is conditional on this brightness and artifact mixture.

</details>

In [ ]:
table([dict(model=k, threshold=v["threshold"], **v["validation"])
       for k,v in reference["baselines"].items()], caption="Measured baselines")

## Ask the experiment to fail in a known way

We permute training labels once with a recorded seed and leave validation
truth untouched. We also deliberately split candidates from one <a class="model-alerts-glossary-term" href="#model-alerts-glossary-field">field</a>
across training and validation: the native <a class="model-alerts-glossary-term" href="#model-alerts-glossary-provenance">provenance</a> guard must reject that
dataset before training. A development-only learning curve uses nested <a class="model-alerts-glossary-term" href="#model-alerts-glossary-field">field</a>
subsets with their own training-only <a class="model-alerts-glossary-term" href="#model-alerts-glossary-normalization">normalization</a>.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

The <a class="model-alerts-glossary-term" href="#model-alerts-glossary-permutation">permutation</a> control tests whether genuine validation structure can be
predicted from randomly assigned training targets. Unexpected predictability
requires diagnosis, not a search for a more reassuring <a class="model-alerts-glossary-term" href="#model-alerts-glossary-permutation">permutation</a> seed.
The invalid <a class="model-alerts-glossary-term" href="#model-alerts-glossary-split">split</a> is a guard demonstration and is never used to claim a
performance gain. Learning curves help identify data limitation but cannot
choose settings after final-test unblinding. Three initialization seeds show
optimization sensitivity; they are not three independent sky surveys.

</details>

In [ ]:
table(dict(invalid_split_rejected=reference["controls"]["invalid_split"]["rejected"],
           permutation_validation_auc=reference["controls"]["permutation"]["validation_roc_auc"],
           permutation_review_required=reference["controls"]["permutation_review_required"],
           cpu_cuda_agreement=reference["controls"]["cpu_cuda_agreement"]),
      caption="Scientific controls")

## Does a measured xFit description add information?

The fusion investigation exports exact <a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">candidate</a> difference stamps and
row-aligned <a class="model-alerts-glossary-term" href="#model-alerts-glossary-variance">variance</a>, fits the Gaussian difference model with xFit, then
builds the native portable feature bundle. <a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">Candidate</a> IDs and stamp hashes
must correspond exactly. Native gates retain failed or invalid fits.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

An ideal single positive source has no negative lobe, while registration
residuals can resemble positive/negative dipoles. Fitted shape, separation,
fit quality and uncertainty can therefore add measured evidence. The
Gaussian fit is an approximation to pixel-integrated sources and imperfect
subtraction. A failed fit is an outcome with <a class="model-alerts-glossary-term" href="#model-alerts-glossary-coverage">coverage</a> implications, not a
reason to discard a difficult <a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">candidate</a>. Report `fit_present`, `fit_valid`
and uncertainty <a class="model-alerts-glossary-term" href="#model-alerts-glossary-coverage">coverage</a>, along with fused classification metrics. Weights
are marginal-variance approximations and omit residual <a class="model-alerts-glossary-term" href="#model-alerts-glossary-covariance">covariance</a>.

</details>

In [ ]:
table(reference["fusion"]["development_coverage"], caption="xFit coverage")
table(reference["fusion"]["validation"]["validation"], caption="Fusion validation")

## Choose a validation operating point with tied scores

For every model and seed, we choose the highest <a class="model-alerts-glossary-term" href="#model-alerts-glossary-threshold">threshold</a> whose inclusive
`score >= <a class="model-alerts-glossary-term" href="#model-alerts-glossary-threshold">threshold</a>` rule retains at least 95% of validation positives.
Checkpoints, <a class="model-alerts-glossary-term" href="#model-alerts-glossary-normalization">normalization</a> and thresholds are frozen before generating or
inferring the final reserved test set. The primary endpoint is test
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-false-positive-rate">false-positive rate</a> at that frozen threshold.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

Sort the $n_+$ positive validation scores in descending order. With
$k=\lceil0.95n_+\rceil$, choose $t=s_{(k)}$. Including every tie at $t$ may
retain more than 95%; raising it would lose required positive examples.

<div class="model-alerts-equation" role="region" aria-label="Scientific equation" tabindex="0">

$$\operatorname{TPR}(t)=\frac{TP}{TP+FN},\qquad
\operatorname{FPR}(t)=\frac{FP}{FP+TN}.$$

</div>

The native training workflow's accuracy-selected <a class="model-alerts-glossary-term" href="#model-alerts-glossary-threshold">threshold</a> is not this
operating point. Test <a class="model-alerts-glossary-term" href="#model-alerts-glossary-recall">recall</a> is an achieved result, not automatically 95%.
Invalidating bugs require a new freeze and fresh reserved seeds; earlier
attempts remain recorded. Exploratory browser sliders do not alter the
frozen endpoint reported in the scientific tables.

</details>

In [ ]:
table({k:reference["freeze"][k] for k in
       ("train_fields","validation_fields","test_fields","shifted_fields",
        "test_seed_block","shift_seed_block")}, caption="Frozen field counts and seed blocks")
table(model_rows(reference["endpoints"]), caption="Frozen-threshold test endpoints")

## Separate detection recovery from classification retention

Some injected changes never pass detection. A classifier cannot recover
those missing candidates. We report detected associated injections divided
by all injected ordinary changes, and final retained associated injections
divided by that same injection denominator. Candidate-level <a class="model-alerts-glossary-term" href="#model-alerts-glossary-recall">recall</a> has a
different denominator: eligible detected positives.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

An injection can create a complicated <a class="model-alerts-glossary-term" href="#model-alerts-glossary-residual">residual</a> and a detection can overlap
multiple causes. Injection recovery is counted by injection identity;
classification confusion counts are counted by <a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">candidate</a> identity. Neither
quantity may silently replace the other. The detection <a class="model-alerts-glossary-term" href="#model-alerts-glossary-threshold">threshold</a> and
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-association">association</a> convention are part of the complete experiment, not fixed facts
of the sky. Coverage tables identify movers, <a class="model-alerts-glossary-term" href="#model-alerts-glossary-ambiguous">ambiguous</a> associations and
binary-eligible candidates for every <a class="model-alerts-glossary-term" href="#model-alerts-glossary-split">split</a>.

</details>

## Read curves and reliability at the modeled prevalence

ROC tracks achieved recall against <a class="model-alerts-glossary-term" href="#model-alerts-glossary-false-positive-rate">false-positive rate</a> as score thresholds
change. Precision–recall makes the class mixture more visible. <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reliability">Reliability</a>
compares <a class="model-alerts-glossary-term" href="#model-alerts-glossary-score">score</a> averages with observed positive fractions in occupied bins;
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-brier-score">Brier score</a> averages squared score errors. We show each initialization seed
rather than hiding outcomes behind a mean.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

<a class="model-alerts-glossary-term" href="#model-alerts-glossary-brier-score">Brier score</a> is

<div class="model-alerts-equation" role="region" aria-label="Scientific equation" tabindex="0">

$$\mathrm{BS}=\frac1N\sum_i(s_i-y_i)^2.$$

</div>

Empty <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reliability">reliability</a> bins contain no evidence and remain empty. A small Brier
score or a visually good <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reliability">reliability</a> curve on this generator does not
calibrate scores for an observed survey. ROC/PR curves and <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reliability">reliability</a> use
all eligible evaluation candidates, not the curated image gallery. The
reported operating point remains the validation-frozen <a class="model-alerts-glossary-term" href="#model-alerts-glossary-threshold">threshold</a>.

</details>

In [ ]:
evaluation = np.load(root / "evaluation.npz", allow_pickle=False)
predictions = {name: evaluation[key] for name, key in reference["score_arrays"].items()}
show(performance_diagnostics(evaluation["labels"], predictions))

## Uncertainty follows independent fields

We bootstrap fields, carrying every <a class="model-alerts-glossary-term" href="#model-alerts-glossary-candidate">candidate</a> and epoch in each sampled
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-field">field</a> together. Pair and triplet differences use the same resampled fields.
The three initialization outcomes remain separate. Intervals are conditional
on trained checkpoints, frozen thresholds and the specified modeled mixture.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

Independent-candidate <a class="model-alerts-glossary-term" href="#model-alerts-glossary-bootstrap">bootstrap</a> would overstate the information in repeated
observations sharing a template. Field <a class="model-alerts-glossary-term" href="#model-alerts-glossary-bootstrap">bootstrap</a> samples field identities
with replacement; repeated draws multiply all their candidates. Replicates
without negatives have undefined FPR and are explicitly counted. The
validation half-width rule is a development <a class="model-alerts-glossary-term" href="#model-alerts-glossary-precision">precision</a> check, not a
post-selection confidence guarantee. These percentile intervals do not
include all training-population, generator-model or threshold-selection
uncertainty.

</details>

In [ ]:
table(reference["uncertainty"], caption="Paired field-bootstrap uncertainty")

## Explore the complete queue, inspect a bounded gallery

Search, <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">reference</a> and signed difference panels share pan, zoom and tapped
pixel selection. Exact raw pixel readouts remain in <a class="model-alerts-glossary-term" href="#model-alerts-glossary-electron">electron</a> units even when
display units change. Navigate candidates or filter by family and confusion
outcome. The gallery contains at most 96 candidates selected by a recorded
round-robin confusion/<a class="model-alerts-glossary-term" href="#model-alerts-glossary-snr">SNR</a>/family rule. Empty categories remain empty.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

A human chooses a review budget as well as a desired completeness. At
assumed positive prevalence $\pi$, holding class-conditional score mixtures
fixed,

<div class="model-alerts-equation" role="region" aria-label="Scientific equation" tabindex="0">

$$\mathrm{purity}=\frac{\pi\,TPR}{\pi\,TPR+(1-\pi)\,FPR},\qquad
N_{\mathrm{queue}}=N[\pi\,TPR+(1-\pi)\,FPR].$$

</div>

The browser recomputes these quantities from complete saved evaluation
scores. It does not retrain or rerun inference. A <a class="model-alerts-glossary-term" href="#model-alerts-glossary-prevalence">prevalence</a> change is not
an arbitrary <a class="model-alerts-glossary-term" href="#model-alerts-glossary-distribution-shift">distribution shift</a>: the projection assumes the positive and
negative subpopulations keep their <a class="model-alerts-glossary-term" href="#model-alerts-glossary-score">score</a> distributions. Gallery counts are
never substituted for population counts. Display transforms and page themes
must leave raw data and equivalent scientific color scales unchanged.

</details>

In [ ]:
gallery = np.load(root / "gallery.npz", allow_pickle=False)
rows = json.loads((root / "gallery-rows.json").read_text())
# The reference stores explicit original gallery indices and the full score pool.
explorer, gallery_rule = candidate_explorer(
    gallery["images"], rows, gallery["labels"], gallery["scores"],
    reference["inspector_threshold"],
    evaluation=dict(labels=evaluation["labels"],
                    scores=evaluation[reference["inspector_score_key"]],
                    snr=evaluation["snr"],
                    candidate_id=evaluation["candidate_id"],
                    family=evaluation["family"]))
show(explorer)
print(json.dumps(reference["gallery_rule"], indent=2))

## Look at high-confidence mistakes

A confident false positive can reveal a subtraction <a class="model-alerts-glossary-term" href="#model-alerts-glossary-residual">residual</a> that resembles
a <a class="model-alerts-glossary-term" href="#model-alerts-glossary-psf">PSF</a>. A confident false negative can reveal source crowding, displacement,
low contrast or a shape outside the training mixture. Inspect errors by
measured <a class="model-alerts-glossary-term" href="#model-alerts-glossary-snr">SNR</a> and artifact family without discarding inconvenient categories.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

Confidence is not correctness. A <a class="model-alerts-glossary-term" href="#model-alerts-glossary-score">score</a> near zero or one may reflect an
unfamiliar example being pushed far from the decision boundary. The
stratified gallery intentionally overrepresents useful error types; it is
not a random sample and must not be used to estimate error rates. Empty
error categories are reported honestly instead of filled with another
model's mistakes. Family/<a class="model-alerts-glossary-term" href="#model-alerts-glossary-snr">SNR</a> tables use the entire eligible evaluation set.

</details>

In [ ]:
table([dict(model=model, family=family, **metrics)
       for model, groups in reference["breakdowns"].items()
       for family, metrics in groups["family"].items()],
      caption="Complete-evaluation family breakdowns", collapsed=True)
table([dict(model=model, measured_snr_bin=snr, **metrics)
       for model, groups in reference["breakdowns"].items()
       for snr, metrics in groups["snr"].items()],
      caption="Complete-evaluation SNR breakdowns", collapsed=True)
table([dict(model=model, **r)
       for model, mistakes in reference["high_confidence_mistakes"].items()
       for r in mistakes], caption="High-confidence mistakes", collapsed=True)

## Try one specified distribution shift

The shift test changes search seeing to $\sigma=1.9$–$2.6$ pixels and
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-background">background</a> to 180–300 electrons per pixel, and adds an unseen column-bleed
artifact. The <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">reference</a> exposure model and classifier thresholds stay fixed.
We report achieved recall, FPR and <a class="model-alerts-glossary-term" href="#model-alerts-glossary-coverage">coverage</a> on these additional fields.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

This shift combines broader sources, larger <a class="model-alerts-glossary-term" href="#model-alerts-glossary-background">background</a> fluctuations and a
new negative family. It can reveal one kind of fragility, but cannot support
a claim of general robustness. Changing <a class="model-alerts-glossary-term" href="#model-alerts-glossary-prevalence">prevalence</a> alone would not create
these image differences. No shifted labels or images choose <a class="model-alerts-glossary-term" href="#model-alerts-glossary-normalization">normalization</a>,
architecture, checkpoints, <a class="model-alerts-glossary-term" href="#model-alerts-glossary-association">association</a> radius or thresholds. Movers remain
a separately identified astrophysical challenge rather than becoming bogus
training labels.

</details>

In [ ]:
table(model_rows(reference["shift"]), caption="Specified shift endpoints")
table([dict(population=population, model=model, **counts)
       for population, results in reference["moving_source_challenge"].items()
       for model, counts in results.items()], caption="Moving-source challenge retention")

## Reproduce the result, then decide what it supports

<a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">Reference</a> receipts record generator settings, seed blocks, source and
artifact hashes, dependencies, hardware descriptions and frozen operating
points. Check them before comparing exports. Training data, weights and
executed notebooks remain outside Git and outside the standalone HTML.
The full <a class="model-alerts-glossary-term" href="#model-alerts-glossary-reference">reference</a> includes all required investigations; quick smoke
exercises APIs and has no scientific performance qualification.

<details open class="model-alerts-science">
<summary>Science Explanation</summary>

After provisioning dependencies, execution must work with network access
blocked. The public HTML contains the bounded gallery and complete evaluation
<a class="model-alerts-glossary-term" href="#model-alerts-glossary-score">score</a>/statistic arrays, not the full training set or checkpoints. Its controls
need no Bokeh server, Python <a class="model-alerts-glossary-term" href="#model-alerts-glossary-kernel">kernel</a>, WebSocket or API. Source ZIPs, executed
notebooks and exported HTML are separate build artifacts. See README and
PUBLISHING for commands and the remaining versioned-Pages integration gap.
Persistent human annotations, if wanted, use the optional local native review
workflow on the newly generated queue. Publication is a later action.

</details>

In [ ]:
table({k:reference["provenance"][k] for k in
       ("cuphoton_revision","hardware","dependencies")}, caption="Execution provenance")
table(reference["artifact_sizes"], caption="Saved artifact bytes")

## Terminology

<dl class="model-alerts-glossary">
<dt id="model-alerts-glossary-psf" tabindex="-1">PSF</dt><dd>Point-spread function: the pixel distribution of an unresolved source; its integral gives source flux.</dd>
<dt id="model-alerts-glossary-reference" tabindex="-1">reference</dt><dd>An earlier noisy exposure shared by repeated search epochs of one independent field.</dd>
<dt id="model-alerts-glossary-search" tabindex="-1">search</dt><dd>The newer exposure inspected for positive astrophysical changes.</dd>
<dt id="model-alerts-glossary-electron" tabindex="-1">electron</dt><dd>The detector count unit here; unit gain equates recorded counts and electrons.</dd>
<dt id="model-alerts-glossary-flux" tabindex="-1">flux</dt><dd>Expected total source electrons integrated over the PSF, before finite-image truncation.</dd>
<dt id="model-alerts-glossary-background" tabindex="-1">background</dt><dd>Expected diffuse electrons per pixel, separate from astronomical source flux.</dd>
<dt id="model-alerts-glossary-seeing" tabindex="-1">seeing</dt><dd>Apparent atmospheric source width, modeled here by a Gaussian sigma in pixels.</dd>
<dt id="model-alerts-glossary-poisson" tabindex="-1">Poisson</dt><dd>Counting distribution whose variance equals its expected electron count.</dd>
<dt id="model-alerts-glossary-read-noise" tabindex="-1">read noise</dt><dd>Independent Gaussian detector readout fluctuation with a declared RMS in electrons.</dd>
<dt id="model-alerts-glossary-kernel" tabindex="-1">kernel</dt><dd>Fitted convolution weights mapping the reference PSF and photometric scale toward the search exposure.</dd>
<dt id="model-alerts-glossary-residual" tabindex="-1">residual</dt><dd>Search minus its fitted matched-reference-plus-background model.</dd>
<dt id="model-alerts-glossary-variance" tabindex="-1">variance</dt><dd>Expected squared fluctuation in the image unit; variance scales quadratically under unit changes.</dd>
<dt id="model-alerts-glossary-covariance" tabindex="-1">covariance</dt><dd>Correlated fluctuations between pixels or observations; omitted from the marginal diagnostic.</dd>
<dt id="model-alerts-glossary-snr" tabindex="-1">SNR</dt><dd>Signal-to-noise ratio; measured detection SNR here is an approximate marginal diagnostic.</dd>
<dt id="model-alerts-glossary-candidate" tabindex="-1">candidate</dt><dd>A measured local positive residual maximum passing the frozen detection rule.</dd>
<dt id="model-alerts-glossary-association" tabindex="-1">association</dt><dd>A truth-only positional rule connecting a measured detection to injected causes.</dd>
<dt id="model-alerts-glossary-ambiguous" tabindex="-1">ambiguous</dt><dd>A detection associated with several genuine changes or a real change overlapping a detector artifact.</dd>
<dt id="model-alerts-glossary-field" tabindex="-1">field</dt><dd>Independent generated sky/detector scene owning its template, sources, defects and repeated epochs.</dd>
<dt id="model-alerts-glossary-epoch" tabindex="-1">epoch</dt><dd>One search observation of a field, with independent search noise.</dd>
<dt id="model-alerts-glossary-split" tabindex="-1">split</dt><dd>Assignment of independent fields to training, validation or reserved final test roles.</dd>
<dt id="model-alerts-glossary-provenance" tabindex="-1">provenance</dt><dd>Recorded origins, identities, labels, seeds and hashes supporting reproducible input relationships.</dd>
<dt id="model-alerts-glossary-normalization" tabindex="-1">normalization</dt><dd>Centering/scaling fitted only on training data and applied unchanged to held-out data.</dd>
<dt id="model-alerts-glossary-logit" tabindex="-1">logit</dt><dd>Unbounded classifier output transformed by the sigmoid to a score.</dd>
<dt id="model-alerts-glossary-score" tabindex="-1">score</dt><dd>Classifier output between zero and one; it need not be a calibrated astrophysical probability.</dd>
<dt id="model-alerts-glossary-checkpoint" tabindex="-1">checkpoint</dt><dd>Saved model parameters selected using development validation loss.</dd>
<dt id="model-alerts-glossary-baseline" tabindex="-1">baseline</dt><dd>A simpler measured-data model used to test whether complex learning adds useful information.</dd>
<dt id="model-alerts-glossary-permutation" tabindex="-1">permutation</dt><dd>A fixed random reassignment of training labels for a negative-control investigation.</dd>
<dt id="model-alerts-glossary-fusion" tabindex="-1">fusion</dt><dd>Combining measured xFit scalar features with image features through native xScan interfaces.</dd>
<dt id="model-alerts-glossary-threshold" tabindex="-1">threshold</dt><dd>The inclusive score cutoff used to retain candidates for review.</dd>
<dt id="model-alerts-glossary-recall" tabindex="-1">recall</dt><dd>Retained eligible detected positives divided by all eligible detected positives.</dd>
<dt id="model-alerts-glossary-false-positive-rate" tabindex="-1">false-positive rate</dt><dd>Retained negatives divided by all eligible negatives, at the specified frozen threshold.</dd>
<dt id="model-alerts-glossary-precision" tabindex="-1">precision</dt><dd>Retained positives divided by all retained candidates; also called purity.</dd>
<dt id="model-alerts-glossary-prevalence" tabindex="-1">prevalence</dt><dd>Fraction of positives in the candidate population before the classifier threshold.</dd>
<dt id="model-alerts-glossary-brier-score" tabindex="-1">Brier score</dt><dd>Mean squared difference between a score and its binary label.</dd>
<dt id="model-alerts-glossary-reliability" tabindex="-1">reliability</dt><dd>Agreement of average scores with observed positive fractions in occupied score bins.</dd>
<dt id="model-alerts-glossary-bootstrap" tabindex="-1">bootstrap</dt><dd>Resampling independent fields with replacement to quantify specified conditional sampling uncertainty.</dd>
<dt id="model-alerts-glossary-distribution-shift" tabindex="-1">distribution shift</dt><dd>A specified change to image or population-generating conditions relative to development.</dd>
<dt id="model-alerts-glossary-coverage" tabindex="-1">coverage</dt><dd>The fraction/count of observations for which a stated method or metric is defined and included.</dd>
<dt id="model-alerts-glossary-cosmic-ray" tabindex="-1">cosmic ray</dt><dd>A modeled compact detector charge event with a morphology distinct from the optical PSF.</dd>
<dt id="model-alerts-glossary-misregistration" tabindex="-1">misregistration</dt><dd>An offset between reference and search source positions that can create dipolar residuals.</dd>
</dl>